# FEM dynamic-winch representative parameter search — v0.15

1100 m FEM cable、pre-launch equilibrium、索重量・索空力・ground contact/friction・reel-in/node removalを含むproduction solver上でStandard/Club representativeを局所探索する。ClubはSecantと同じSOP parameterを使うことを原則とする。


In [ ]:
from copy import deepcopy
from pathlib import Path
import json

import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
from winch_launch_simulator.simulation import load_case, run_simulation


def set_value(case, path, value):
    target = case
    for key in path[:-1]:
        target = target[key]
    target[path[-1]] = float(value)


def event(result, code):
    return next((item for item in result["events"] if item["code"] == code), None)


def measure_case(case, dt_s):
    candidate = deepcopy(case)
    candidate["simulation"]["dt_s"] = dt_s
    result = run_simulation(candidate)
    history = result["history"]
    liftoff = event(result, "LIFTOFF")
    release = event(result, "PILOT_COMMAND_RELEASE")
    stops = [item for item in result["events"] if item["severity"] == "stop"]
    warnings = [item for item in result["events"] if item["severity"] == "warning"]
    if stops or liftoff is None or release is None:
        return {"feasible": False, "reason": stops[0]["code"] if stops else "MISSING_LIFTOFF_OR_RELEASE"}

    time = np.asarray(history["time_s"], dtype=float)
    agl = np.asarray(history["agl_m"], dtype=float)
    eas = np.asarray(history["eas_m_s"], dtype=float)
    q = np.asarray(history["pitch_rate_deg_s"], dtype=float)
    pitch = np.asarray(history["theta_deg"], dtype=float)
    tension = np.asarray(history["cable_tension_n"], dtype=float)
    stall = np.asarray(history["stall_margin_m_s"], dtype=float)
    phase = np.asarray(history["pilot_phase"], dtype=object)
    elevator = np.asarray(history["elevator_deg"], dtype=float)

    liftoff_index = int(np.argmin(np.abs(time - liftoff["time"])))
    release_index = int(np.argmin(np.abs(time - release["time"])))
    rotation_indices = np.flatnonzero(phase == "PROGRESSIVE_ROTATION")
    full_climb = phase == "FULL_CLIMB"
    if not len(rotation_indices) or not np.any(full_climb):
        return {"feasible": False, "reason": "MISSING_PILOT_PHASE"}
    rotation_index = int(rotation_indices[0])
    elevator_rate = np.diff(elevator) / np.diff(time)

    metrics = {
        "feasible": len(warnings) == 0,
        "reason": "OK" if not warnings else warnings[0]["code"],
        "warning_count": len(warnings),
        "liftoff_s": float(liftoff["time"]),
        "rotation_start_agl_m": float(agl[rotation_index]),
        "max_eas_m_s": float(np.max(eas[liftoff_index : release_index + 1])),
        "max_eas_after_200ft_m_s": float(np.max(eas[(agl >= 60.96) & (time <= release["time"])])),
        "max_tension_n": float(np.max(tension[liftoff_index : release_index + 1])),
        "min_stall_margin_m_s": float(np.min(stall[liftoff_index : release_index + 1])),
        "max_abs_pitch_rate_deg_s": float(np.max(np.abs(q[liftoff_index : release_index + 1]))),
        "elevator_rate_rms_deg_s": float(np.sqrt(np.mean(elevator_rate**2))),
        "release_altitude_m": float(agl[release_index]),
        "release_eas_m_s": float(eas[release_index]),
        "release_q_deg_s": abs(float(q[release_index])),
        "initialization": result["metadata"]["initialization"],
        "fem_max_ground_penetration_m": float(np.max(history["fem_max_ground_penetration_m"])),
        "node_removals": result["metadata"]["fem_nodes_removed"],
    }

    if candidate["pilot"]["sop"] == "standard":
        target_eas = candidate["pilot"]["full_climb"]["target_eas_m_s"]
        metrics["full_climb_eas_rms_m_s"] = float(np.sqrt(np.mean((eas[full_climb] - target_eas) ** 2)))
    else:
        target_pitch = candidate["pilot"]["full_climb"]["pitch_target_deg"]
        metrics["full_climb_pitch_rms_error_deg"] = float(np.sqrt(np.mean((pitch[full_climb] - target_pitch) ** 2)))
        metrics["power_reduction_requested"] = event(result, "WINCH_POWER_REDUCTION_REQUEST") is not None
        tail = np.asarray(history["tailwheel_normal_force_n"], dtype=float)
        between = (time >= liftoff["time"] - 4.0) & (time <= liftoff["time"])
        metrics["tailwheel_positive_fraction_near_liftoff"] = float(np.mean(tail[between] > 0.0)) if np.any(between) else np.nan

    return metrics


def local_sweep(base, dt_s, sweep):
    rows = []
    for path, values in sweep.items():
        for value in values:
            candidate = deepcopy(base)
            set_value(candidate, path, value)
            metrics = measure_case(candidate, dt_s)
            rows.append({"parameter": ".".join(path), "value": value, **metrics})
    return rows

STANDARD_PATH = ROOT / "cases" / "g103a_dynamic_winch_fem.json"
CLUB_PATH = ROOT / "cases" / "g103a_club_sop_dynamic_winch_fem.json"
OUTPUT_DIR = ROOT / "outputs" / "parameter_search"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

standard_base = load_case(STANDARD_PATH)
club_base = load_case(CLUB_PATH)
SEARCH_DT_S = 0.005
VALIDATION_DT_S = 0.0025


def fem_feasible(metrics):
    if not metrics.get("feasible", False):
        return False
    initialization = metrics["initialization"]
    return (
        metrics["min_stall_margin_m_s"] >= 1.50
        and metrics["max_abs_pitch_rate_deg_s"] <= 8.0
        and metrics["max_tension_n"] < 7000.0
        and metrics["fem_max_ground_penetration_m"] <= 0.005
        and initialization["static_max_free_node_force_residual_n"] <= 1.0
        and abs(initialization["initial_winch_tension_n"] - 2500.0) <= 50.0
    )


## 1. Canonical FEM feasibility

まず現在の20-node canonicalを評価する。Clubは200 ft transition、45°command、Vw protection、LEVEL OFF/LOS releaseまでを含めて判定する。


In [ ]:
standard_metrics = measure_case(standard_base, SEARCH_DT_S)
club_metrics = measure_case(club_base, SEARCH_DT_S)
print("standard feasible:", fem_feasible(standard_metrics))
print(json.dumps(standard_metrics, ensure_ascii=False, indent=2, default=str))
print("club feasible:", fem_feasible(club_metrics))
print("club rotation AGL:", club_metrics.get("rotation_start_agl_m"))
print(json.dumps(club_metrics, ensure_ascii=False, indent=2, default=str))


## 2. Standard SOP local sensitivity

FEM physical parametersは固定し、Standard controller / target tensionだけを局所探索する。


In [ ]:
STANDARD_SWEEP = {
    ("pilot", "pitch_rate_gain_s"): [2.35, 2.40, 2.45],
    ("pilot", "initial_climb", "pitch_target_deg"): [19.0, 20.0, 21.0],
    ("pilot", "rotation", "start_eas_m_s"): [26.2, 26.5, 26.8],
    ("pilot", "rotation", "min_duration_s"): [7.2, 7.4, 7.6],
    ("pilot", "full_climb", "target_eas_m_s"): [27.2, 27.5, 27.8],
    ("pilot", "full_climb", "eas_kp"): [0.70, 0.75, 0.80],
    ("winch", "control", "target_tension_schedule", "max_tension_n"): [4800.0, 5000.0, 5200.0],
}
standard_trials = local_sweep(standard_base, SEARCH_DT_S, STANDARD_SWEEP)
[row for row in standard_trials if fem_feasible(row)][:12]


## 3. Club common-SOP local sensitivity

Secantと別pilotをfitしない。共通Club candidateのうちFEM stall margin / pitch-rate / weak-link余裕を支配するparameterを重点的に振る。


In [ ]:
CLUB_SWEEP = {
    ("pilot", "ground_roll", "tailwheel_load_cue_n"): [400.0, 500.0, 600.0],
    ("pilot", "hold_down", "low_tension_cue_n"): [3600.0, 4000.0, 4400.0],
    ("pilot", "hold_down", "low_tension_relief_gain_deg"): [16.0, 20.0, 24.0],
    ("pilot", "rotation", "duration_s"): [7.0, 7.4, 7.8],
    ("pilot", "full_climb", "pitch_kp"): [0.45, 0.50, 0.55],
    ("pilot", "full_climb", "elevator_bias_deg"): [-4.5, -4.0, -3.5],
    ("pilot", "full_climb", "speed_protection_margin_m_s"): [1.3, 1.5, 1.7],
    ("pilot", "full_climb", "power_reduction_request_margin_m_s"): [0.7, 0.9, 1.1],
}
club_trials = local_sweep(club_base, SEARCH_DT_S, CLUB_SWEEP)
[row for row in club_trials if fem_feasible(row) and row.get("rotation_start_agl_m", 0.0) >= 60.7][:12]


## 4. Numerical V&V and candidate export

20-node canonicalのtime-step refinementと20/22/40-node mesh sensitivityはcontroller searchから分離して扱う。v0.18のfull-launch numerical V&Vは `validation/reference/fem_convergence.json` をrelease recordとし、launch-power transient固有のv0.17記録はhistorical/change-specific referenceとして残す。


In [ ]:
for candidate in (standard_base, club_base):
    metrics = measure_case(candidate, VALIDATION_DT_S)
    print(candidate["case_id"], "feasible:", fem_feasible(metrics))
    print(json.dumps(metrics, ensure_ascii=False, indent=2, default=str))

for candidate in (standard_base, club_base):
    output = deepcopy(candidate)
    output["case_id"] += "_candidate"
    path = OUTPUT_DIR / f"{output['case_id']}.json"
    path.write_text(json.dumps(output, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(path)


## 5. FEM physical sensitivity is separate from controller search

`node_count`、cable-ground stiffness/damping/friction、cable aerodynamic drag、winch torque-speed curveはcontroller optimizer変数にしない。これらはmodel/parameter validationとして独立に扱う。
